# load dataset

In [1]:
import pandas as pd
dataset = pd.read_csv("processed_ai4i2020.csv")
dataset.head()
dataset[dataset['Machine failure']==1]

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,Temp_diff [K],power,wear_rate
50,1,298.9,309.1,2861,4.6,143,1,10.2,13160.6,0.049983
69,1,298.9,309.0,1410,65.7,191,1,10.1,92637.0,0.135461
77,1,298.8,308.9,1455,41.3,208,1,10.1,60091.5,0.142955
160,1,298.4,308.2,1282,60.7,216,1,9.8,77817.4,0.168487
161,1,298.3,308.1,1412,52.3,218,1,9.8,73847.6,0.154391
...,...,...,...,...,...,...,...,...,...,...
9758,1,298.6,309.8,2271,16.2,218,1,11.2,36790.2,0.095993
9764,1,298.5,309.5,1294,66.7,12,1,11.0,86309.8,0.009274
9822,1,298.5,309.4,1360,60.9,187,1,10.9,82824.0,0.137500
9830,1,298.3,309.3,1337,56.1,206,1,11.0,75005.7,0.154076


In [2]:
dataset.columns

Index(['Type', 'Air temperature [K]', 'Process temperature [K]',
       'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]',
       'Machine failure', 'Temp_diff [K]', 'power', 'wear_rate'],
      dtype='object')

# input output split

In [3]:
x=dataset.drop(['Machine failure'],axis=1)
y=dataset['Machine failure']

In [4]:
x.head()

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Temp_diff [K],power,wear_rate
0,2,298.1,308.6,1551,42.8,0,10.5,66382.8,0.000000
1,1,298.2,308.7,1408,46.3,3,10.5,65190.4,0.002131
2,1,298.1,308.5,1498,49.4,5,10.4,74001.2,0.003338
3,1,298.2,308.6,1433,39.5,7,10.4,56603.5,0.004885
4,1,298.2,308.7,1408,40.0,9,10.5,56320.0,0.006392


In [5]:
y.head()

0    0
1    0
2    0
3    0
4    0
Name: Machine failure, dtype: int64

# train test split

In [6]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.30,random_state=42)

In [7]:
for i in y_train, y_test:
    counts = pd.DataFrame(i).value_counts()
    print(counts)
# y_train -> is imbalanced - handled using SMOTE

Machine failure
0                  6754
1                   246
Name: count, dtype: int64
Machine failure
0                  2907
1                    93
Name: count, dtype: int64


# standardization

In [8]:
# using Robust Scaler --> Keeps the exact extreme values but scales the "normal" data around them.
from sklearn.preprocessing import RobustScaler
transformer = RobustScaler()
x_train_scaled= transformer.fit_transform(x_train)
x_test_scaled = transformer.transform(x_test)
x_train_scaled, x_test_scaled

(array([[ 1.        , -0.928     , -0.82608696, ...,  0.70588235,
         -0.93571061,  0.13383271],
        [ 1.        , -0.256     , -0.39130435, ...,  0.05882353,
          0.12315128, -0.66475612],
        [ 1.        ,  0.128     ,  0.82608696, ...,  1.        ,
          1.21199846,  0.65180413],
        ...,
        [-1.        ,  0.864     ,  0.95652174, ..., -0.17647059,
          0.71977015,  1.14748581],
        [-1.        , -1.28      , -1.39130435, ...,  0.58823529,
         -0.712176  , -0.69075272],
        [ 0.        ,  0.032     ,  0.13043478, ...,  0.23529412,
         -0.97233353, -0.72006417]]),
 array([[ 0.        ,  0.224     ,  0.08695652, ..., -0.17647059,
         -0.32919327,  0.83912351],
        [ 1.        ,  1.12      ,  0.73913043, ..., -0.94117647,
          0.26058507,  0.01716729],
        [ 1.        , -0.576     , -0.95652174, ..., -0.11764706,
          0.16704951,  0.12708813],
        ...,
        [ 0.        , -0.032     , -0.43478261, ..., -

## save the standardization file 

In [10]:
import pickle 
pickle.dump(transformer, open('robust_scaler.sav','wb'))

# handling imbalanced data - for training set

In [11]:
from imblearn.over_sampling import SMOTE    # need to be applied only in training data - to avoid data leakage
smote = SMOTE(random_state=42)
x_train_resampled, y_train_resampled = smote.fit_resample(x_train_scaled, y_train)

In [12]:
y_train_resampled.value_counts()

Machine failure
0    6754
1    6754
Name: count, dtype: int64

# Model Creation

In [13]:
# model prediction & evaluation 
def cm_pred_eval(classifier,x_test,y_test):
    # prediction
    y_pred=classifier.predict(x_test)
    # confusion matrix
    from sklearn.metrics import confusion_matrix
    cm=confusion_matrix(y_test,y_pred)
    # classification report 
    from sklearn.metrics import classification_report
    classi_report=classification_report(y_test,y_pred)
    # accuracy 
    from sklearn.metrics import accuracy_score
    accuracy=accuracy_score(y_test,y_pred)
    # ROC AUC score
    from sklearn.metrics import roc_auc_score
    roc = roc_auc_score(y_test, y_pred)
    return classifier,accuracy,roc,cm,classi_report,x_test,y_test

# model creation 
def train_and_evaluate(model,x_train,y_train,x_test,y_test):
    model.fit(x_train, y_train)
    return cm_pred_eval(model,x_test,y_test)


In [14]:
# imports
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier

In [15]:
# calling models
classifier,accuracy,roc,cm,classi_report,x_test,y_test = train_and_evaluate(LogisticRegression(random_state=0), 
                                                                        x_train_resampled,y_train_resampled,x_test_scaled,y_test)
print(f"\n\n MODEL: {classifier}\n ACCURACY: {accuracy}\n ROC: {roc}\n CONFUSION MATRIX: \n{cm}\n CLASSIFICATION REPORT: \n{classi_report}\n")

classifier,accuracy,roc,cm,classi_report,x_test,y_test = train_and_evaluate(KNeighborsClassifier(n_neighbors=5,p=2, metric='minkowski'),
                                                                        x_train_resampled,y_train_resampled,x_test_scaled,y_test)
print(f"\n\n MODEL: {classifier}\n ACCURACY: {accuracy}\n ROC: {roc}\n CONFUSION MATRIX: \n{cm}\n CLASSIFICATION REPORT: \n{classi_report}\n")

classifier,accuracy,roc,cm,classi_report,x_test,y_test = train_and_evaluate(SVC(kernel='linear',random_state=0),
                                                                     x_train_resampled,y_train_resampled,x_test_scaled,y_test)
print(f"\n\n MODEL: {classifier}\n ACCURACY: {accuracy}\n ROC: {roc}\n CONFUSION MATRIX: \n{cm}\n CLASSIFICATION REPORT: \n{classi_report}\n")

classifier,accuracy,roc,cm,classi_report,x_test,y_test = train_and_evaluate(SVC(kernel='rbf',random_state=0),
                                                                        x_train_resampled,y_train_resampled,x_test_scaled,y_test)
print(f"\n\n MODEL: {classifier}\n ACCURACY: {accuracy}\n ROC: {roc}\n CONFUSION MATRIX: \n{cm}\n CLASSIFICATION REPORT: \n{classi_report}\n")

classifier,accuracy,roc,cm,classi_report,x_test,y_test = train_and_evaluate(GaussianNB(),x_train_resampled,y_train_resampled,x_test_scaled,y_test)
print(f"\n\n MODEL: {classifier}\n ACCURACY: {accuracy}\n ROC: {roc}\n CONFUSION MATRIX: \n{cm}\n CLASSIFICATION REPORT: \n{classi_report}\n")

classifier,accuracy,roc,cm,classi_report,x_test,y_test = train_and_evaluate(DecisionTreeClassifier(criterion='entropy',random_state=0),
                                                                        x_train_resampled,y_train_resampled,x_test_scaled,y_test)
print(f"\n\n MODEL: {classifier}\n ACCURACY: {accuracy}\n ROC: {roc}\n CONFUSION MATRIX: \n{cm}\n CLASSIFICATION REPORT: \n{classi_report}\n")

classifier,accuracy,roc,cm,classi_report,x_test,y_test = train_and_evaluate(RandomForestClassifier(n_estimators=100,criterion='entropy',random_state=0),
                                                                        x_train_resampled,y_train_resampled,x_test_scaled,y_test)
print(f"\n\n MODEL: {classifier}\n ACCURACY: {accuracy}\n ROC: {roc}\n CONFUSION MATRIX: \n{cm}\n CLASSIFICATION REPORT: \n{classi_report}\n")

classifier,accuracy,roc,cm,classi_report,x_test,y_test = train_and_evaluate(GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,
    max_depth=3, random_state=0), x_train_resampled,y_train_resampled,x_test_scaled,y_test)
print(f"\n\n MODEL: {classifier}\n ACCURACY: {accuracy}\n ROC: {roc}\n CONFUSION MATRIX: \n{cm}\n CLASSIFICATION REPORT: \n{classi_report}\n")



 MODEL: LogisticRegression(random_state=0)
 ACCURACY: 0.868
 ROC: 0.8382103265754519
 CONFUSION MATRIX: 
[[2529  378]
 [  18   75]]
 CLASSIFICATION REPORT: 
              precision    recall  f1-score   support

           0       0.99      0.87      0.93      2907
           1       0.17      0.81      0.27        93

    accuracy                           0.87      3000
   macro avg       0.58      0.84      0.60      3000
weighted avg       0.97      0.87      0.91      3000




 MODEL: KNeighborsClassifier()
 ACCURACY: 0.9373333333333334
 ROC: 0.8427599675976786
 CONFUSION MATRIX: 
[[2743  164]
 [  24   69]]
 CLASSIFICATION REPORT: 
              precision    recall  f1-score   support

           0       0.99      0.94      0.97      2907
           1       0.30      0.74      0.42        93

    accuracy                           0.94      3000
   macro avg       0.64      0.84      0.70      3000
weighted avg       0.97      0.94      0.95      3000




 MODEL: SVC(kernel='lin

## Model tuning - for Top 3 models (SVC non linear, Random forest, GradientBoosting)

In [16]:
# DONE BEFORE FEATURE ENGINEERING 

# from sklearn.model_selection import GridSearchCV
# from sklearn.svm import SVC
# param_grid_svc = {
#     'C': [0.1, 1, 10],
#     'gamma': ['scale', 'auto'],
#     'kernel': ["poly", "rbf", "sigmoid"]
# }
# grid_svc = GridSearchCV(
#     SVC(),
#     param_grid_svc,
#     cv=5,
#     scoring='f1_weighted',
#     n_jobs=-1,
#     refit=True,
#     verbose=3
# )
# # model creation
# grid_svc.fit(x_train_resampled, y_train_resampled)
# print("Best Params SVC:", grid_svc.best_params_)
# # model evaluation
# classifier,accuracy,roc,cm,classi_report,x_test,y_test = cm_pred_eval(grid_svc,x_test,y_test)
# print(f"\n\n MODEL: {classifier}\n ACCURACY: {accuracy}\n ROC: {roc}\n CONFUSION MATRIX: \n{cm}\n CLASSIFICATION REPORT: \n{classi_report}\n")

In [ ]:
# FIRST BEST MODEL FROM model training without tuining
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
param_grid_rf = {
    "criterion": ["gini", "entropy", "log_loss"],
    "class_weight":["balanced", "balanced_subsample"],
    'n_estimators': [100, 200],
    'max_depth': [None, 5, 10]
}
grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=0),
    param_grid_rf,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    refit=True,
    verbose=3
)
# model creation
grid_rf.fit(x_train_resampled, y_train_resampled)
print("Best Params RF:", grid_rf.best_params_)
# model evaluation
classifier,accuracy,roc,cm,classi_report,x_test,y_test = cm_pred_eval(grid_rf,x_test,y_test)
print(f"\n\n MODEL: {classifier}\n ACCURACY: {accuracy}\n ROC: {roc}\n CONFUSION MATRIX: \n{cm}\n CLASSIFICATION REPORT: \n{classi_report}\n")

Fitting 5 folds for each of 36 candidates, totalling 180 fits


In [ ]:
# SECOND BEST MODEL FROM model training without tuining
from sklearn.tree import DecisionTreeClassifier
param_grid_decision = {
    "criterion": ["gini", "entropy", "log_loss"],
    'max_depth': [None, 5, 10, 20],         # Prevent overfitting
    'min_samples_split': [2, 5, 10]       # Minimum samples to split a node
}
grid_decision = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid_decision,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    refit=True,
    verbose=3
)
# model creation
grid_decision.fit(x_train_resampled, y_train_resampled)
print("Best Params RF:", grid_decision.best_params_)
# model evaluation
classifier,accuracy,roc,cm,classi_report,x_test,y_test = cm_pred_eval(grid_decision,x_test,y_test)
print(f"\n\n MODEL: {classifier}\n ACCURACY: {accuracy}\n ROC: {roc}\n CONFUSION MATRIX: \n{cm}\n CLASSIFICATION REPORT: \n{classi_report}\n")

In [ ]:
from imblearn.ensemble import BalancedRandomForestClassifier
classifier,accuracy,roc,cm,classi_report,x_test,y_test = train_and_evaluate(BalancedRandomForestClassifier(random_state=42),
                                                                        x_train_resampled,y_train_resampled,x_test_scaled,y_test)
print(f"\n\n MODEL: {classifier}\n ACCURACY: {accuracy}\n ROC: {roc}\n CONFUSION MATRIX: \n{cm}\n CLASSIFICATION REPORT: \n{classi_report}\n")

# Final Model

In [ ]:
grid_rf

"I finalized Random Forest because it gave the best balance between detecting failures (recall 0.78) and avoiding false alarms (precision 0.57). It handles feature correlations well and is robust to the imbalanced nature of the dataset. Metrics like ROC-AUC 0.88 and accuracy 97% indicate strong model performance."

## Save the Model 

In [ ]:
import pickle
pickle.dump(grid_rf, open("grid_rf_final_model.sav","wb"))

In [ ]:
x.info()

In [ ]:
y_pred = grid_rf.predict(x_test_scaled)
y_pred[0:20]

In [ ]:
y_test[0:20]